# Ingest results.json file Assignment


## Assignment
- create a DF loading the constructors.json file in the raw container
- rename columns to (race_id,driver_id)
- add a new columns ingestion_timestampt
- save the file in parquet file in the processed container.
- drop statusId
- verify the schema of the partquet file

In [0]:
% run "../includes/common_functions"

In [0]:
%run "../includes/configuration"

In [0]:
dbutils.widgets.text("p_date_source","")
#dbutils.widgets.dropdown("p_date_source","Testing",["Testing","Production"])
v_data_source=dbutils.widgets.get("p_date_source")

In [0]:
from pyspark.sql.functions import current_timestamp,col


In [0]:
display(spark.read.option("multiLine",True).json(f"{raw_folder_path}t/pit_stops.json").summary())





In [0]:
pit_stops_schema="raceId INT,driverId INT,stop STRING,lap INT,time STRING,duration DOUBLE,milliseconds INT"


In [0]:
pits_stops_df =spark.read.option("multiLine",True).json(f"{raw_folder_path}/pit_stops.json")


In [0]:
pits_stops_df =pits_stops_df.withColumnRenamed("raceId","race_id").withColumnRenamed("driverId","driver_id")

In [0]:
pit_stops_df=add_ingestion_timestamp(pits_stops_df)
pit_stops_df = add_data_source(pit_stops_df,v_data_source)

In [0]:
display(pits_stops_df)

## Write DF into parquet file

In [0]:
pits_stops_df.write.mode("overwrite").partitionBy("race_id").parquet(f"{processed_folder_path}/pits_stops")

In [0]:
"df=spark.read.parquet(f"{processed_folder_path}/pits_stops")
"df.printSchema()

In [0]:
dbutils.notebook.exit("Success")